# Medical RecSys and Chatbot
In this assignment I used a medical dataset for which I built items using various NLP methods like TF-IDF, BoW and embeddings, and ranking them using text and signals like rating, effectiveness and side effects. AFter building the recsys I also built a chatbot using a Gemini API which turns a conversation into recommendations.

In [ ]:
!pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

In [ ]:
import pandas as pd

In [ ]:
drug_reviews = fetch_ucirepo(id=461)

X = drug_reviews.data.features
y = drug_reviews.data.targets

In [ ]:
X.head()

,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


X contains all of the data, while y is empty (None). Each row is review, not drug-specific. Since I will be building an item-to-item recsys, I will later aggregate the reviews into one text.

In [ ]:
df = X.copy()

In [ ]:
df.head()

,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview,commentsReview
0,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ...","monitor blood pressure , weight and asses for ..."
1,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon...","I Hate This Birth Control, I Would Not Suggest..."
2,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.,I took 2 pills at the onset of my menstrual cr...
3,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines...",I was given Prilosec prescription at a dose of...
4,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...,See above


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4143 entries, 0 to 4142
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   urlDrugName        4143 non-null   object
 1   rating             4143 non-null   int64 
 2   effectiveness      4143 non-null   object
 3   sideEffects        4143 non-null   object
 4   condition          4142 non-null   object
 5   benefitsReview     4120 non-null   object
 6   sideEffectsReview  4045 non-null   object
 7   commentsReview     4130 non-null   object
dtypes: int64(1), object(7)
memory usage: 259.1+ KB


There are 8 columns out of which only 1 (rating) is numeric, urlDrugName which is the name of the drug, rating which is on a 1-10 scale,

In [ ]:
df.isnull().sum()

,0
urlDrugName,0
rating,0
effectiveness,0
sideEffects,0
condition,1
benefitsReview,23
sideEffectsReview,98
commentsReview,13


In [ ]:
print("Rating range:")
print(df["rating"].describe())

print("\nEffectiveness labels:")
print(df["effectiveness"].value_counts(dropna=False))

print("\nSide-effects labels:")
print(df["sideEffects"].value_counts(dropna=False))


Rating range:
count    4143.000000
mean        6.946416
std         2.948868
min         1.000000
25%         5.000000
50%         8.000000
75%         9.000000
max        10.000000
Name: rating, dtype: float64

Effectiveness labels:
effectiveness
Highly Effective          1741
Considerably Effective    1238
Moderately Effective       572
Ineffective                329
Marginally Effective       263
Name: count, dtype: int64

Side-effects labels:
sideEffects
Mild Side Effects                1349
No Side Effects                  1198
Moderate Side Effects             850
Severe Side Effects               491
Extremely Severe Side Effects     255
Name: count, dtype: int64


This diagnostic cell helps verify the rating scale and the categorical labels for effectiveness and side-effect severity before using them in reranking.

The missingness is very low and understandable as not every drug has to have side effects. The only row I decided to drop is the one lacking the condition because it is one of the key identifiers. I filled the other missing values with "" so vectorizers can process it properly.

In [ ]:
df = df.dropna(subset = ["condition"]).copy()

In [ ]:
text_cols = ["benefitsReview", "sideEffectsReview", "commentsReview"]
df[text_cols] = df[text_cols].fillna("")

Although models like TF-IDF, Bag of Words and embedding models expect a single document of text per row, if I just blindly merged the columns it could cause problems. So instead I transformed each of the benefits, side effects and comments to vectors and then computed the final similarity.

In [ ]:
from collections import Counter

def mode_or_unknown(series):
    values = [str(v).strip() for v in series if pd.notna(v) and str(v).strip()]
    if not values:
        return "Unknown"
    return Counter(values).most_common(1)[0][0]

item_df = (
    df.groupby(["urlDrugName", "condition"], as_index=False)
      .agg(
          benefitsReview=("benefitsReview", lambda x: " ".join([t for t in x if t.strip()])),
          sideEffectsReview=("sideEffectsReview", lambda x: " ".join([t for t in x if t.strip()])),
          commentsReview=("commentsReview", lambda x: " ".join([t for t in x if t.strip()])),
          rating=("rating", "mean"),
          review_count=("rating", "size"),
          effectiveness=("effectiveness", mode_or_unknown),
          sideEffects=("sideEffects", mode_or_unknown),
      )
)


In [ ]:
item_df["rating"] = item_df["rating"].round(2)

print("Original rows:", df.shape)
print("Item catalog shape:", item_df.shape)

item_df[[
    "urlDrugName", "condition", "rating", "review_count", "effectiveness", "sideEffects"
]].head()


Original rows: (4142, 8)
Item catalog shape: (2655, 9)


,urlDrugName,condition,rating,review_count,effectiveness,sideEffects
0,abilify,bipolar,5.0,2,Highly Effective,Moderate Side Effects
1,abilify,bipolar disorder,3.0,1,Marginally Effective,Severe Side Effects
2,abilify,depression,6.0,3,Considerably Effective,No Side Effects
3,abilify,depression not resolved with antidepressant drugs,2.0,1,Considerably Effective,Extremely Severe Side Effects
4,abilify,depression/anxiety,10.0,1,Highly Effective,No Side Effects


I decided to merge all the rows with the same drug name and condition into one row, without doing this the recsys would consider each review as if the rated item is a unique item. Basically every row becomes one drug for one condition. This creates a richer representation per drug-condition pair.

In [ ]:
import re
import pandas as pd

def normalize_condition(text):
    if pd.isna(text):
        return ""

    s = str(text).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    if s in {"htn", "high blood pressure", "hypertension"}:
        return "hypertension"
    if "congestive heart failure" in s or "heart failure" in s:
        return "heart failure"
    if s in {"atrial fib", "a fib", "atrial fibrillation"}:
        return "atrial fibrillation"
    if s in {"high cholesterol", "hypercholesterolemia", "hyperlipidemia"}:
        return "hyperlipidemia"

    return s

item_df["condition_norm"] = item_df["condition"].apply(normalize_condition)


In [ ]:
item_df[["condition", "condition_norm"]].drop_duplicates().sort_values("condition").head(50)

,condition,condition_norm
1776,1mg,1mg
180,2,2
1319,2 broken arms,2 broken arms
2263,2 compressed discs in neck,2 compressed discs in neck
387,20 year pack a day smoker,20 year pack a day smoker
722,75 mg,75 mg
1747,? 'heart failure',heart failure
144,a boil,a boil
925,a little bit of osteoporosis in the hips,a little bit of osteoporosis in the hips
1480,a typical migraines,a typical migraines


I normalized some of the most common conditions so that the same condition is not recognized as different ones, like in the case of hypertension.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

benefits_vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
sideeffects_vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
comments_vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

benefits_tfidf = benefits_vectorizer.fit_transform(item_df["benefitsReview"])
sideeffects_tfidf = sideeffects_vectorizer.fit_transform(item_df["sideEffectsReview"])
comments_tfidf = comments_vectorizer.fit_transform(item_df["commentsReview"])

sim_benefits = cosine_similarity(benefits_tfidf)
sim_sideeffects = cosine_similarity(sideeffects_tfidf)
sim_comments = cosine_similarity(comments_tfidf)

In [ ]:
w_benefits = 0.5
w_sideeffects = 0.3
w_comments = 0.2

final_sim = (
    w_benefits * sim_benefits +
    w_sideeffects * sim_sideeffects +
    w_comments * sim_comments
)

I have created three separate TF-IDF vectorizers and gave them different weights, prioritizing the benefits and side effects, while the comments are least important. I did this because review columns don't cover the same things, so my model doesn't just compute the cosine similarity for the text but creates a weighted blend.

In [ ]:
def recommend_by_item(drug_name, condition, item_df, sim_matrix, top_n=5):
    matches = item_df[
        (item_df["urlDrugName"].str.lower() == drug_name.lower()) &
        (item_df["condition"].str.lower() == condition.lower())
    ]

    if matches.empty:
        return f"No item found for drug='{drug_name}' and condition='{condition}'."

    idx = matches.index[0]

    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # skip itself
    sim_scores = sim_scores[1:top_n+1]

    rec_indices = [i for i, _ in sim_scores]
    rec_scores = [score for _, score in sim_scores]

    results = item_df.iloc[rec_indices][["urlDrugName", "condition", "rating"]].copy()
    results["similarity_score"] = rec_scores

    return results.reset_index(drop=True)

I first created a plain item to item recsys which takes a drug, a condition and a similarity matrix and returns the most similar items

In [ ]:
recommend_by_item("enalapril", "management of congestive heart failure", item_df, final_sim, top_n=5)

,urlDrugName,condition,rating,similarity_score
0,coreg,cardiomyopathy,10.0,0.144115
1,nuvaring,heavy period,1.0,0.116249
2,vasotec,htn,9.0,0.114156
3,atenolol,mitral valve prolapse,10.0,0.101322
4,atenolol,atrial fibrillation,1.0,0.095219


After this first test I realized the recommendations were still too noisy for a health-related project. The main issues were generic lexical matches, condition variants such as `htn` versus `hypertension`, and the fact that low-rated items could still surface because rating and effectiveness were not yet part of the ranking logic.

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []
    for word in text.split():
        if word not in stop_words and len(word) > 2:
            tokens.append(lemmatizer.lemmatize(word))

    return " ".join(tokens)

for col in ["benefitsReview", "sideEffectsReview", "commentsReview"]:
    item_df[f"{col}_clean"] = item_df[col].apply(preprocess_text)

item_df[[
    "urlDrugName", "condition",
    "benefitsReview_clean", "sideEffectsReview_clean", "commentsReview_clean"
]].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,urlDrugName,condition,benefitsReview_clean,sideEffectsReview_clean,commentsReview_clean
0,abilify,bipolar,severe depression agitation mixed state bipola...,became drowsy however adequate sleep hr hangov...,abilify decreased need daily klonopin went tap...
1,abilify,bipolar disorder,notice benefit supposedly help keep mood balan...,uncomfortable inner restlessness worst side ef...,prescribed abilify daily assist moderating moo...
2,abilify,depression,week noticed change already awake seems though...,none far none far none far,doctor added abilify cymbalta feeling really f...
3,abilify,depression not resolved with antidepressant drugs,abilify honestly say depression resolved,caused memory loss incident involving retail t...,taking ativan getting psychological therapy tr...
4,abilify,depression/anxiety,within week taking cocktail abilify lexapro ex...,side effect noticed,take one pill thing feeling better ever


I lowercased the text, removed punctuation and non-letter characters, normalized whitespaces, tokenized, removed stop words, did lemmatization and finally joined it back into normal text.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

benefits_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    max_features=8000
)

sideeffects_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    max_features=8000
)

comments_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    max_features=8000
)

benefits_tfidf = benefits_vectorizer.fit_transform(item_df["benefitsReview_clean"])
sideeffects_tfidf = sideeffects_vectorizer.fit_transform(item_df["sideEffectsReview_clean"])
comments_tfidf = comments_vectorizer.fit_transform(item_df["commentsReview_clean"])

sim_benefits = cosine_similarity(benefits_tfidf)
sim_sideeffects = cosine_similarity(sideeffects_tfidf)
sim_comments = cosine_similarity(comments_tfidf)

In [ ]:
w_benefits = 0.60
w_sideeffects = 0.15
w_comments = 0.25

final_sim = (
    w_benefits * sim_benefits +
    w_sideeffects * sim_sideeffects +
    w_comments * sim_comments
)

Then I built an improved TF-IDF model on the cleaned text. Since a lot of medical terms consist of 2 words, I decided to use both uni and bigrams.

In [ ]:
import numpy as np

effectiveness_rank = {
    "Ineffective": 0,
    "Marginally Effective": 1,
    "Moderately Effective": 2,
    "Considerably Effective": 3,
    "Highly Effective": 4,
}

side_effects_rank = {
    "No Side Effects": 0,
    "Mild Side Effects": 1,
    "Moderate Side Effects": 2,
    "Severe Side Effects": 3,
    "Extremely Severe Side Effects": 4,
}

def recommend_by_item_advanced(
    drug_name,
    condition,
    item_df,
    sim_matrix,
    top_n=10,
    min_rating=7.0,
    min_effectiveness_score=2,
    max_side_effects_score=2,
    condition_bonus=0.25,
    rating_bonus=0.10,
):
    condition_norm = normalize_condition(condition)

    matches = item_df[
        (item_df["urlDrugName"].str.lower() == drug_name.lower()) &
        (item_df["condition_norm"] == condition_norm)
    ]

    if matches.empty:
        return f"No item found for drug='{drug_name}' and condition='{condition}'."

    idx = matches.index[0]
    seed_condition_norm = item_df.loc[idx, "condition_norm"]

    candidates = item_df.copy()
    candidates["base_similarity"] = sim_matrix[idx]
    candidates = candidates.drop(index=idx)

    candidates = candidates[candidates["condition_norm"] == seed_condition_norm]

    candidates["effectiveness_score"] = candidates["effectiveness"].map(effectiveness_rank)
    candidates["side_effects_score"] = candidates["sideEffects"].map(side_effects_rank)

    candidates = candidates[candidates["rating"] >= min_rating]

    if min_effectiveness_score is not None:
        candidates = candidates[
            candidates["effectiveness_score"].fillna(-1) >= min_effectiveness_score
        ]

    if max_side_effects_score is not None:
        candidates = candidates[
            candidates["side_effects_score"].fillna(99) <= max_side_effects_score
        ]

    candidates["condition_bonus"] = (
        candidates["condition_norm"] == seed_condition_norm
    ).astype(float) * condition_bonus

    candidates["rating_bonus"] = (
        (candidates["rating"] - min_rating) / max(10 - min_rating, 1e-9)
    ).clip(lower=0, upper=1) * rating_bonus

    candidates["final_score"] = (
        candidates["base_similarity"]
        + candidates["condition_bonus"]
        + candidates["rating_bonus"]
    )

    return (
        candidates.sort_values("final_score", ascending=False)
        [[
            "urlDrugName", "condition", "condition_norm", "rating", "review_count",
            "effectiveness", "sideEffects", "base_similarity", "condition_bonus",
            "rating_bonus", "final_score"
        ]]
        .head(top_n)
        .reset_index(drop=True)
    )

# Backwards-compatible alias in case older cells still call the original name
recommend_by_item_reranked = recommend_by_item_advanced


I mapped the effectiveness into numerical scores and sideEffects into numeric severity.

In [ ]:
recommend_by_item_advanced(
    "cozaar",
    "high blood pressure",
    item_df,
    final_sim,
    top_n=10,
    min_rating=5.0,
    min_effectiveness_score=2,
    max_side_effects_score=3,
)


,urlDrugName,condition,condition_norm,rating,review_count,effectiveness,sideEffects,base_similarity,condition_bonus,rating_bonus,final_score
0,tenormin,high blood pressure,hypertension,10.00,2,Highly Effective,No Side Effects,0.359356,0.25,0.1000,0.709356
1,dyrenium,hypertension,hypertension,10.00,1,Considerably Effective,Moderate Side Effects,0.335793,0.25,0.1000,0.685793
2,vasotec,hypertension,hypertension,10.00,1,Highly Effective,No Side Effects,0.279722,0.25,0.1000,0.629722
3,hyzaar,high blood pressure,hypertension,7.80,5,Highly Effective,Mild Side Effects,0.308388,0.25,0.0560,0.614388
4,tekturna,hypertension,hypertension,10.00,1,Highly Effective,Mild Side Effects,0.225543,0.25,0.1000,0.575543
5,prinivil,high blood pressure,hypertension,5.00,13,Moderately Effective,Mild Side Effects,0.320393,0.25,0.0000,0.570393
6,sular,high blood pressure,hypertension,10.00,1,Highly Effective,No Side Effects,0.192870,0.25,0.1000,0.542870
7,diovan,high blood pressure,hypertension,7.17,6,Highly Effective,Moderate Side Effects,0.246198,0.25,0.0434,0.539598
8,benicar,high blood pressure,hypertension,9.00,1,Considerably Effective,No Side Effects,0.199972,0.25,0.0800,0.529972
9,metoprolol,hypertension,hypertension,8.00,1,Considerably Effective,No Side Effects,0.214759,0.25,0.0600,0.524759


The results seem pretty good bearing in mind that the condition is hypertension in all of the rows, most have high ratings, good effectiveness, mild or no side effects

In [ ]:
%pip install -U sentence-transformers



In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embed_model = SentenceTransformer("all-MiniLM-L6-v2")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


I installed sentence-transformers and encoded each text using MiniLM, so instead of representing text as counts I represented them as dense semantic vector embeddings. This is important because they can capture semantic similarity instead of looking for an exact match.

In [ ]:
benefits_emb = embed_model.encode(
    item_df["benefitsReview"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

sideeffects_emb = embed_model.encode(
    item_df["sideEffectsReview"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

comments_emb = embed_model.encode(
    item_df["commentsReview"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

sim_benefits_emb = cosine_similarity(benefits_emb)
sim_sideeffects_emb = cosine_similarity(sideeffects_emb)
sim_comments_emb = cosine_similarity(comments_emb)

final_sim_emb = (
    0.60 * sim_benefits_emb +
    0.15 * sim_sideeffects_emb +
    0.25 * sim_comments_emb
)

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

Batches:   0%|          | 0/83 [00:00<?, ?it/s]

The embedding-based recommender keeps the same multi-channel architecture, but replaces TF-IDF with sentence embeddings so that similarity is based more on meaning than exact word overlap.

In [ ]:
recommend_by_item_advanced(
    "cozaar",
    "high blood pressure",
    item_df,
    final_sim_emb,
    top_n=10,
    min_rating=7.0,
    min_effectiveness_score=2,
    max_side_effects_score=3,
)


,urlDrugName,condition,condition_norm,rating,review_count,effectiveness,sideEffects,base_similarity,condition_bonus,rating_bonus,final_score
0,tekturna,hypertension,hypertension,10.0,1,Highly Effective,Mild Side Effects,0.586109,0.25,0.100000,0.936109
1,tenormin,high blood pressure,hypertension,10.0,2,Highly Effective,No Side Effects,0.578879,0.25,0.100000,0.928879
2,vasotec,hypertension,hypertension,10.0,1,Highly Effective,No Side Effects,0.535606,0.25,0.100000,0.885606
3,atenolol,hypertension,hypertension,9.0,2,Highly Effective,Mild Side Effects,0.552202,0.25,0.066667,0.868869
4,benicar,hypertension,hypertension,10.0,1,Highly Effective,No Side Effects,0.509897,0.25,0.100000,0.859897
5,zebeta,hypertension,hypertension,9.0,1,Considerably Effective,Mild Side Effects,0.538397,0.25,0.066667,0.855063
6,metoprolol,hypertension,hypertension,8.0,1,Considerably Effective,No Side Effects,0.566109,0.25,0.033333,0.849443
7,dyrenium,hypertension,hypertension,10.0,1,Considerably Effective,Moderate Side Effects,0.497235,0.25,0.100000,0.847235
8,sular,high blood pressure,hypertension,10.0,1,Highly Effective,No Side Effects,0.482365,0.25,0.100000,0.832365
9,benicar,high blood pressure,hypertension,9.0,1,Considerably Effective,No Side Effects,0.478747,0.25,0.066667,0.795414


When we compare the recommendations for the embeddings and TF-IDF recsys, although the TF-IDF one was really good, the embeddings one seems to have a slightly better performance. The top items are highly effective, highly rated and have mild or no side effects.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

benefits_bow_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    max_features=8000
)

sideeffects_bow_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    max_features=8000
)

comments_bow_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    max_features=8000
)

benefits_bow = benefits_bow_vectorizer.fit_transform(item_df["benefitsReview_clean"])
sideeffects_bow = sideeffects_bow_vectorizer.fit_transform(item_df["sideEffectsReview_clean"])
comments_bow = comments_bow_vectorizer.fit_transform(item_df["commentsReview_clean"])

sim_benefits_bow = cosine_similarity(benefits_bow)
sim_sideeffects_bow = cosine_similarity(sideeffects_bow)
sim_comments_bow = cosine_similarity(comments_bow)

final_sim_bow = (
    0.60 * sim_benefits_bow +
    0.15 * sim_sideeffects_bow +
    0.25 * sim_comments_bow
)

print(final_sim_bow.shape)

(2655, 2655)


I also created a Bag-of-Words based model so it can serve as a simple less nuanced baseline for comparison.

In [ ]:
recommend_by_item_advanced(
    "cozaar",
    "high blood pressure",
    item_df,
    final_sim_bow,
    top_n=10
)

,urlDrugName,condition,condition_norm,rating,review_count,effectiveness,sideEffects,base_similarity,condition_bonus,rating_bonus,final_score
0,tenormin,high blood pressure,hypertension,10.00,2,Highly Effective,No Side Effects,0.500016,0.25,0.100000,0.850016
1,dyrenium,hypertension,hypertension,10.00,1,Considerably Effective,Moderate Side Effects,0.400032,0.25,0.100000,0.750032
2,vasotec,hypertension,hypertension,10.00,1,Highly Effective,No Side Effects,0.388626,0.25,0.100000,0.738626
3,tekturna,hypertension,hypertension,10.00,1,Highly Effective,Mild Side Effects,0.360803,0.25,0.100000,0.710803
4,hyzaar,high blood pressure,hypertension,7.80,5,Highly Effective,Mild Side Effects,0.412153,0.25,0.026667,0.688820
5,metoprolol,hypertension,hypertension,8.00,1,Considerably Effective,No Side Effects,0.370883,0.25,0.033333,0.654216
6,diovan,high blood pressure,hypertension,7.17,6,Highly Effective,Moderate Side Effects,0.383105,0.25,0.005667,0.638772
7,sular,high blood pressure,hypertension,10.00,1,Highly Effective,No Side Effects,0.287338,0.25,0.100000,0.637338
8,benicar,high blood pressure,hypertension,9.00,1,Considerably Effective,No Side Effects,0.319862,0.25,0.066667,0.636529
9,avapro,hypertension,hypertension,10.00,1,Highly Effective,No Side Effects,0.241666,0.25,0.100000,0.591666


In [ ]:
def compare_methods(drug_name, condition, top_n=5):
    print("=== BoW ===")
    display(recommend_by_item_advanced(drug_name, condition, item_df, final_sim_bow, top_n=top_n))

    print("\n=== TF-IDF ===")
    display(recommend_by_item_advanced(drug_name, condition, item_df, final_sim, top_n=top_n))

    print("\n=== Embeddings ===")
    display(recommend_by_item_advanced(drug_name, condition, item_df, final_sim_emb, top_n=top_n))

In [ ]:
compare_methods("cozaar", "high blood pressure", top_n=5)

=== BoW ===


,urlDrugName,condition,condition_norm,rating,review_count,effectiveness,sideEffects,base_similarity,condition_bonus,rating_bonus,final_score
0,tenormin,high blood pressure,hypertension,10.0,2,Highly Effective,No Side Effects,0.500016,0.25,0.100000,0.850016
1,dyrenium,hypertension,hypertension,10.0,1,Considerably Effective,Moderate Side Effects,0.400032,0.25,0.100000,0.750032
2,vasotec,hypertension,hypertension,10.0,1,Highly Effective,No Side Effects,0.388626,0.25,0.100000,0.738626
3,tekturna,hypertension,hypertension,10.0,1,Highly Effective,Mild Side Effects,0.360803,0.25,0.100000,0.710803
4,hyzaar,high blood pressure,hypertension,7.8,5,Highly Effective,Mild Side Effects,0.412153,0.25,0.026667,0.688820



=== TF-IDF ===


,urlDrugName,condition,condition_norm,rating,review_count,effectiveness,sideEffects,base_similarity,condition_bonus,rating_bonus,final_score
0,tenormin,high blood pressure,hypertension,10.0,2,Highly Effective,No Side Effects,0.359356,0.25,0.100000,0.709356
1,dyrenium,hypertension,hypertension,10.0,1,Considerably Effective,Moderate Side Effects,0.335793,0.25,0.100000,0.685793
2,vasotec,hypertension,hypertension,10.0,1,Highly Effective,No Side Effects,0.279722,0.25,0.100000,0.629722
3,hyzaar,high blood pressure,hypertension,7.8,5,Highly Effective,Mild Side Effects,0.308388,0.25,0.026667,0.585054
4,tekturna,hypertension,hypertension,10.0,1,Highly Effective,Mild Side Effects,0.225543,0.25,0.100000,0.575543



=== Embeddings ===


,urlDrugName,condition,condition_norm,rating,review_count,effectiveness,sideEffects,base_similarity,condition_bonus,rating_bonus,final_score
0,tekturna,hypertension,hypertension,10.0,1,Highly Effective,Mild Side Effects,0.586109,0.25,0.100000,0.936109
1,tenormin,high blood pressure,hypertension,10.0,2,Highly Effective,No Side Effects,0.578879,0.25,0.100000,0.928879
2,vasotec,hypertension,hypertension,10.0,1,Highly Effective,No Side Effects,0.535606,0.25,0.100000,0.885606
3,atenolol,hypertension,hypertension,9.0,2,Highly Effective,Mild Side Effects,0.552202,0.25,0.066667,0.868869
4,benicar,hypertension,hypertension,10.0,1,Highly Effective,No Side Effects,0.509897,0.25,0.100000,0.859897


As we can see in the tables above, the embeddings based recsys achieves the highest final score, which makes sense as it uses a different method than BoW and TF-IDF, as they look for a basic overlap and embeddings do it using cosine similarity.

In [ ]:
from IPython.display import display
from itertools import combinations
import json
import os

item_df = item_df.reset_index(drop=True).copy()
item_df["item_id"] = item_df.index
item_df["effectiveness_score"] = item_df["effectiveness"].map(effectiveness_rank)
item_df["side_effects_score"] = item_df["sideEffects"].map(side_effects_rank)

review_df = df.copy()
review_df["condition_norm"] = review_df["condition"].apply(normalize_condition)
for col in text_cols:
    review_df[f"{col}_clean"] = review_df[col].fillna("").apply(preprocess_text)
review_df["user_profile_text"] = (
    review_df["condition_norm"].fillna("") + " " +
    review_df["benefitsReview_clean"].fillna("") + " " +
    review_df["commentsReview_clean"].fillna("") + " " +
    review_df["sideEffectsReview_clean"].fillna("")
).str.replace(r"\s+", " ", regex=True).str.strip()
review_df = review_df.merge(
    item_df[["urlDrugName", "condition", "condition_norm", "item_id"]],
    on=["urlDrugName", "condition", "condition_norm"],
    how="inner"
)
review_df["effectiveness_score"] = review_df["effectiveness"].map(effectiveness_rank)
review_df["side_effects_score"] = review_df["sideEffects"].map(side_effects_rank)

weight_configs = {
    "text_only": {"text": (0.60, 0.15, 0.25), "structured": 0.00},
    "balanced_hybrid": {"text": (0.55, 0.15, 0.20), "structured": 0.10},
    "safety_first": {"text": (0.45, 0.20, 0.15), "structured": 0.20},
}
condition_aliases = {
    "hypertension": ["high blood pressure", "hypertension", "blood pressure", "htn"],
    "heart failure": ["heart failure", "congestive heart failure", "chf"],
    "atrial fibrillation": ["atrial fibrillation", "atrial fib", "a fib"],
    "hyperlipidemia": ["hyperlipidemia", "high cholesterol", "cholesterol", "lipids"],
}

cf_df = df.copy()
cf_df["condition_norm"] = cf_df["condition"].apply(normalize_condition)
cf_df["rating_bucket"] = pd.cut(cf_df["rating"], bins=[0, 4, 7, 10], labels=["low", "medium", "high"], include_lowest=True)
cf_df["profile_id"] = (
    cf_df["condition_norm"].fillna("unknown") + "|" +
    cf_df["effectiveness"].fillna("Unknown") + "|" +
    cf_df["sideEffects"].fillna("Unknown") + "|" +
    cf_df["rating_bucket"].astype(str)
)
profile_drug = pd.crosstab(cf_df[cf_df["rating"] >= 7]["profile_id"], cf_df[cf_df["rating"] >= 7]["urlDrugName"])
cf_drugs = profile_drug.columns.tolist()
cf_lookup = {d: i for i, d in enumerate(cf_drugs)}
cf_sim = cosine_similarity(profile_drug.T) if cf_drugs else np.empty((0, 0))

def relevant_items(case):
    eff = 2 if pd.isna(case["effectiveness_score"]) else max(1, int(case["effectiveness_score"]) - 1)
    side = 2 if pd.isna(case["side_effects_score"]) else min(3, int(case["side_effects_score"]) + 1)
    rel = item_df[
        (item_df["condition_norm"] == case["condition_norm"]) &
        (item_df["item_id"] != case["item_id"]) &
        (item_df["rating"] >= max(7.0, case["rating"] - 1.0)) &
        (item_df["effectiveness_score"].fillna(-1) >= eff) &
        (item_df["side_effects_score"].fillna(99) <= side)
    ]["item_id"].tolist()
    if rel:
        return rel
    return item_df[
        (item_df["condition_norm"] == case["condition_norm"]) &
        (item_df["item_id"] != case["item_id"])
    ].sort_values(["rating", "review_count"], ascending=[False, False])["item_id"].head(10).tolist()

eval_cases = review_df[(review_df["rating"] >= 8) & (review_df["user_profile_text"].str.len() >= 20)].copy()
eval_cases["relevant_items"] = eval_cases.apply(relevant_items, axis=1)
eval_cases = eval_cases[eval_cases["relevant_items"].map(len) > 0].copy()
if len(eval_cases) > 200:
    eval_cases = eval_cases.sample(200, random_state=42).sort_index()

def parse_preferences(text, condition=None, rating_hint=7.0, eff_hint=2, side_hint=2):
    low = str(text).lower()
    found = condition
    for norm, aliases in condition_aliases.items():
        if any(a in low for a in aliases):
            found = norm
            break
    if found is None:
        found = item_df["condition_norm"].mode().iloc[0]
    return {
        "condition": found,
        "min_rating": 8.0 if any(p in low for p in ["effectiveness matters", "works reliably", "highly effective", "very effective"]) else float(rating_hint),
        "min_eff": 3 if any(p in low for p in ["highly effective", "very effective", "most effective"]) else int(2 if pd.isna(eff_hint) else eff_hint),
        "max_side": 1 if any(p in low for p in ["avoid severe side effects", "mild or no side effects", "avoid side effects", "low side effects"]) else int(2 if pd.isna(side_hint) else side_hint),
    }

def score_query(query, model_name="tfidf", weights=(0.55, 0.15, 0.20)):
    wb, ws, wc = weights
    if model_name == "bow":
        q = preprocess_text(query) or "treatment medicine symptom relief"
        return (
            wb * cosine_similarity(benefits_bow_vectorizer.transform([q]), benefits_bow).ravel() +
            ws * cosine_similarity(sideeffects_bow_vectorizer.transform([q]), sideeffects_bow).ravel() +
            wc * cosine_similarity(comments_bow_vectorizer.transform([q]), comments_bow).ravel()
        )
    if model_name == "embedding":
        if "embed_model" in globals() and embed_model is not None:
            q = str(query).strip() or "treatment medicine symptom relief"
            emb = embed_model.encode([q], show_progress_bar=False, convert_to_numpy=True)
            return (
                wb * cosine_similarity(emb, benefits_emb).ravel() +
                ws * cosine_similarity(emb, sideeffects_emb).ravel() +
                wc * cosine_similarity(emb, comments_emb).ravel()
            )
        model_name = "tfidf"
    q = preprocess_text(query) or "treatment medicine symptom relief"
    return (
        wb * cosine_similarity(benefits_vectorizer.transform([q]), benefits_tfidf).ravel() +
        ws * cosine_similarity(sideeffects_vectorizer.transform([q]), sideeffects_tfidf).ravel() +
        wc * cosine_similarity(comments_vectorizer.transform([q]), comments_tfidf).ravel()
    )

def recommend_query(query, model_name="tfidf", condition=None, rating_hint=7.0, eff_hint=2, side_hint=2, exclude_item_id=None, top_n=10, text_weights=(0.55, 0.15, 0.20), structured_weight=0.10):
    prefs = parse_preferences(query, condition=condition, rating_hint=rating_hint, eff_hint=eff_hint, side_hint=side_hint)
    cand = item_df.copy()
    cand["base_score"] = score_query(query, model_name=model_name, weights=text_weights)
    if exclude_item_id is not None:
        cand = cand[cand["item_id"] != exclude_item_id]
    same = cand[cand["condition_norm"] == prefs["condition"]].copy()
    if not same.empty:
        cand = same
    cand["condition_bonus"] = (cand["condition_norm"] == prefs["condition"]).astype(float) * (0.10 + structured_weight)
    cand["rating_bonus"] = ((cand["rating"] - prefs["min_rating"]) / max(10 - prefs["min_rating"], 1e-9)).clip(0, 1) * (0.08 + 0.40 * structured_weight)
    cand["effectiveness_bonus"] = (cand["effectiveness_score"].fillna(-1) >= prefs["min_eff"]).astype(float) * (0.05 + 0.35 * structured_weight)
    cand["safety_bonus"] = (cand["side_effects_score"].fillna(99) <= prefs["max_side"]).astype(float) * (0.05 + 0.35 * structured_weight)
    cand["final_score"] = cand["base_score"] + cand["condition_bonus"] + cand["rating_bonus"] + cand["effectiveness_bonus"] + cand["safety_bonus"]
    return cand.sort_values(["final_score", "rating", "review_count"], ascending=[False, False, False])[["item_id", "urlDrugName", "condition", "rating", "review_count", "effectiveness", "sideEffects", "final_score"]].head(top_n).reset_index(drop=True)

def recommend_popular(case, top_n=10):
    return item_df[
        (item_df["condition_norm"] == case["condition_norm"]) &
        (item_df["item_id"] != case["item_id"])
    ].sort_values(["rating", "review_count"], ascending=[False, False])[["item_id", "urlDrugName", "condition", "rating", "review_count", "effectiveness", "sideEffects"]].head(top_n).reset_index(drop=True)

def recommend_cf(case, top_n=10):
    cand = item_df[(item_df["condition_norm"] == case["condition_norm"]) & (item_df["item_id"] != case["item_id"])].copy()
    if case["urlDrugName"] not in cf_lookup or cf_sim.size == 0:
        return recommend_popular(case, top_n)
    scores = pd.Series(cf_sim[cf_lookup[case["urlDrugName"]]], index=cf_drugs)
    cand["cf_score"] = cand["urlDrugName"].map(scores).fillna(0)
    return cand.sort_values(["cf_score", "rating", "review_count"], ascending=[False, False, False])[["item_id", "urlDrugName", "condition", "rating", "review_count", "effectiveness", "sideEffects", "cf_score"]].head(top_n).reset_index(drop=True)

def diversity(ids, sim_matrix):
    if len(ids) < 2:
        return np.nan
    return 1 - float(np.mean([sim_matrix[i, j] for i, j in combinations(ids, 2)]))

def precision_at_k(rec, rel, k): return len(set(rec[:k]) & set(rel)) / k if k else 0.0
def recall_at_k(rec, rel, k): return len(set(rec[:k]) & set(rel)) / len(set(rel)) if rel else 0.0
def hit_rate_at_k(rec, rel, k): return float(len(set(rec[:k]) & set(rel)) > 0)
def reciprocal_rank_at_k(rec, rel, k):
    for rank, item_id in enumerate(rec[:k], start=1):
        if item_id in rel:
            return 1.0 / rank
    return 0.0
def average_precision_at_k(rec, rel, k):
    hits, total, rel_set = 0, 0.0, set(rel)
    for rank, item_id in enumerate(rec[:k], start=1):
        if item_id in rel_set:
            hits += 1
            total += hits / rank
    return total / min(len(rel_set), k) if hits else 0.0

methods = {
    "Popularity baseline": lambda case: recommend_popular(case, 10),
    "Collaborative proxy baseline": lambda case: recommend_cf(case, 10),
    "BoW hybrid": lambda case: recommend_query(case["user_profile_text"], "bow", condition=case["condition_norm"], rating_hint=max(7.0, case["rating"] - 1.0), eff_hint=case["effectiveness_score"], side_hint=case["side_effects_score"], exclude_item_id=case["item_id"], top_n=10, text_weights=weight_configs["balanced_hybrid"]["text"], structured_weight=weight_configs["balanced_hybrid"]["structured"]),
    "TF-IDF hybrid": lambda case: recommend_query(case["user_profile_text"], "tfidf", condition=case["condition_norm"], rating_hint=max(7.0, case["rating"] - 1.0), eff_hint=case["effectiveness_score"], side_hint=case["side_effects_score"], exclude_item_id=case["item_id"], top_n=10, text_weights=weight_configs["balanced_hybrid"]["text"], structured_weight=weight_configs["balanced_hybrid"]["structured"]),
    "Embedding hybrid": lambda case: recommend_query(case["user_profile_text"], "embedding", condition=case["condition_norm"], rating_hint=max(7.0, case["rating"] - 1.0), eff_hint=case["effectiveness_score"], side_hint=case["side_effects_score"], exclude_item_id=case["item_id"], top_n=10, text_weights=weight_configs["balanced_hybrid"]["text"], structured_weight=weight_configs["balanced_hybrid"]["structured"]),
}
div_mats = {"Popularity baseline": final_sim, "Collaborative proxy baseline": final_sim, "BoW hybrid": final_sim_bow, "TF-IDF hybrid": final_sim, "Embedding hybrid": final_sim_emb}

rows = []
for name, fn in methods.items():
    scores = []
    for _, case in eval_cases.iterrows():
        recs = fn(case)
        ids = recs["item_id"].tolist()
        rel = case["relevant_items"]
        scores.append({
            "precision@10": precision_at_k(ids, rel, 10),
            "recall@10": recall_at_k(ids, rel, 10),
            "hit_rate@10": hit_rate_at_k(ids, rel, 10),
            "mrr@10": reciprocal_rank_at_k(ids, rel, 10),
            "map@10": average_precision_at_k(ids, rel, 10),
            "diversity@10": diversity(ids, div_mats[name]),
        })
    row = pd.DataFrame(scores).mean(numeric_only=True).to_dict()
    row["method"] = name
    rows.append(row)
evaluation_results = pd.DataFrame(rows)[["method", "precision@10", "recall@10", "hit_rate@10", "mrr@10", "map@10", "diversity@10"]].sort_values(["map@10", "hit_rate@10", "diversity@10"], ascending=False).reset_index(drop=True)

weight_rows = []
for name, cfg in weight_configs.items():
    scores = []
    for _, case in eval_cases.iterrows():
        recs = recommend_query(case["user_profile_text"], "tfidf", condition=case["condition_norm"], rating_hint=max(7.0, case["rating"] - 1.0), eff_hint=case["effectiveness_score"], side_hint=case["side_effects_score"], exclude_item_id=case["item_id"], top_n=10, text_weights=cfg["text"], structured_weight=cfg["structured"])
        ids, rel = recs["item_id"].tolist(), case["relevant_items"]
        scores.append({
            "precision@10": precision_at_k(ids, rel, 10),
            "recall@10": recall_at_k(ids, rel, 10),
            "hit_rate@10": hit_rate_at_k(ids, rel, 10),
            "mrr@10": reciprocal_rank_at_k(ids, rel, 10),
            "map@10": average_precision_at_k(ids, rel, 10),
            "diversity@10": diversity(ids, final_sim),
        })
    row = pd.DataFrame(scores).mean(numeric_only=True).to_dict()
    row["configuration"] = name
    weight_rows.append(row)
weight_results = pd.DataFrame(weight_rows)[["configuration", "precision@10", "recall@10", "hit_rate@10", "mrr@10", "map@10", "diversity@10"]].sort_values(["map@10", "diversity@10"], ascending=False).reset_index(drop=True)

display(evaluation_results)
display(weight_results)

scenarios = [
    {"scenario": "Exact lexical overlap", "query": "I need a heart failure treatment that helps with swelling and shortness of breath and I can tolerate only mild side effects.", "condition": "heart failure"},
    {"scenario": "Paraphrased semantic request", "query": "I want something that keeps my blood pressure stable and makes me feel normal during the day.", "condition": "hypertension"},
    {"scenario": "Safety-sensitive request", "query": "I need help with cholesterol but I am worried about strong side effects and want the safest reliable option.", "condition": "hyperlipidemia"},
]
scenario_rows = []
for s in scenarios:
    scenario_rows.append({
        "scenario": s["scenario"],
        "BoW top-1": recommend_query(s["query"], "bow", condition=s["condition"], top_n=1).iloc[0]["urlDrugName"],
        "TF-IDF top-1": recommend_query(s["query"], "tfidf", condition=s["condition"], top_n=1).iloc[0]["urlDrugName"],
        "Embedding top-1": recommend_query(s["query"], "embedding", condition=s["condition"], top_n=1).iloc[0]["urlDrugName"],
    })
display(pd.DataFrame(scenario_rows))

,method,precision@10,recall@10,hit_rate@10,mrr@10,map@10,diversity@10
0,Popularity baseline,0.4020,0.948236,1.00,0.997500,0.987647,0.897234
1,Embedding hybrid,0.3360,0.851693,0.98,0.938500,0.806143,0.512810
2,TF-IDF hybrid,0.3405,0.866939,0.98,0.899333,0.789356,0.873237
3,BoW hybrid,0.3280,0.850557,0.98,0.850345,0.749737,0.802763
4,Collaborative proxy baseline,0.3135,0.829769,0.97,0.798839,0.687441,0.872377


,configuration,precision@10,recall@10,hit_rate@10,mrr@10,map@10,diversity@10
0,safety_first,0.3480,0.877267,0.98,0.947917,0.840595,0.878606
1,balanced_hybrid,0.3405,0.866939,0.98,0.899333,0.789356,0.873237
2,text_only,0.3215,0.845644,0.98,0.827387,0.715529,0.863319


,scenario,BoW top-1,TF-IDF top-1,Embedding top-1
0,Exact lexical overlap,enalapril,enalapril,enalapril
1,Paraphrased semantic request,tenormin,tenormin,tekturna
2,Safety-sensitive request,crestor,crestor,crestor


In the cell above I defined several recommendation models, 2 baselines which are the collaborative proxy one and popularity based, as well as the three hybrid content based ones that use TF-IDF, BoW and embeddings. To evaluate them I created a set of relevant items for each case and compared the models using the metrics we covered in class like precision, recall, hit rate, MRR, MAP and diversity.

From the tables above I concluded that the popularity based recsys performs the best overall which suggests that the highly rated drugs in the dataset within the same condition are a very strong predictor of relevance. Among the content based recsys, the embeddings one performed the best in terms of MAP and MRR, but the TF-IDF one beats it in regards to the diversity. As expected the collaborative proxy performed the worst because we don't have actual user histories and is just an approximation.

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
manual_conversation = [
    {"speaker": "Customer", "text": "I need something for high blood pressure."},
    {"speaker": "Assistant", "text": "What matters most to you: strong effectiveness, low side effects, or lower cost?"},
    {"speaker": "Customer", "text": "Effectiveness matters most, but I want mild or no side effects if possible."},
    {"speaker": "Assistant", "text": "Do you prefer well-rated and reliable medicines?"},
    {"speaker": "Customer", "text": "Yes, I prefer medicines that are well reviewed and reliable."},
]
manual_text = " ".join(t["text"] for t in manual_conversation if t["speaker"] == "Customer")
manual_recommendations = recommend_query(manual_text, "embedding", condition="hypertension", top_n=5)
display(pd.DataFrame(manual_conversation))
display(manual_recommendations)

def fallback_chat(customer_need, reason="Gemini API unavailable"):
    return {
        "conversation": [
            {"speaker": "Assistant", "text": "What condition are you treating and what matters most to you?"},
            {"speaker": "Customer", "text": customer_need},
            {"speaker": "Assistant", "text": "How strict should I be about side effects?"},
            {"speaker": "Customer", "text": "Please prefer mild or no side effects and good effectiveness."},
            {"speaker": "Assistant", "text": "Should I also prioritize strong ratings and reliability?"},
            {"speaker": "Customer", "text": "Yes, prioritize well-reviewed reliable options."},
        ],
        "backend": "template_fallback",
        "reason": reason,
    }

def generate_chat(customer_need):
    import os, json
    from google import genai

    api_key = os.getenv("GEMINI_API_KEY")
    model_name = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

    if not api_key:
        return fallback_chat(customer_need, "GEMINI_API_KEY is not set")

    prompt = f"""
You are a cautious pharmacy chatbot.

Task:
- Ask exactly 3 short follow-up questions.
- Also answer those questions as the customer would.
- Return ONLY valid JSON.

Output format:
{{
  "conversation": [
    {{"speaker": "Assistant", "text": "..."}},
    {{"speaker": "Customer", "text": "..."}},
    {{"speaker": "Assistant", "text": "..."}},
    {{"speaker": "Customer", "text": "..."}},
    {{"speaker": "Assistant", "text": "..."}},
    {{"speaker": "Customer", "text": "..."}}
  ]
}}

Customer need: {customer_need}
"""

    try:
        client = genai.Client(api_key=api_key)

        response = client.models.generate_content(
            model=model_name,
            contents=prompt,
            config={
                "response_mime_type": "application/json",
                "response_json_schema": {
                    "type": "object",
                    "properties": {
                        "conversation": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "speaker": {"type": "string"},
                                    "text": {"type": "string"}
                                },
                                "required": ["speaker", "text"]
                            }
                        }
                    },
                    "required": ["conversation"]
                }
            }
        )

        payload = json.loads(response.text)
        payload["backend"] = model_name
        return payload

    except Exception as e:
        return fallback_chat(
            customer_need,
            f"Gemini call failed: {type(e).__name__}: {e}"
        )




,speaker,text
0,Customer,I need something for high blood pressure.
1,Assistant,What matters most to you: strong effectiveness...
2,Customer,"Effectiveness matters most, but I want mild or..."
3,Assistant,Do you prefer well-rated and reliable medicines?
4,Customer,"Yes, I prefer medicines that are well reviewed..."


,item_id,urlDrugName,condition,rating,review_count,effectiveness,sideEffects,final_score
0,2208,tekturna,hypertension,10.0,1,Highly Effective,Mild Side Effects,1.010545
1,533,cozaar,high blood pressure,10.0,2,Highly Effective,No Side Effects,0.954597
2,216,atenolol,high blood pressure,10.0,1,Highly Effective,No Side Effects,0.953190
3,2380,vasotec,hypertension,10.0,1,Highly Effective,No Side Effects,0.927701
4,1492,norvasc,htn,10.0,1,Highly Effective,No Side Effects,0.925283


The combination of the chatbot and recommendation system seems to be working properly as the conversation identifies the 3 key preferences: the target condition, the most important thing being effectiveness and that the user prefers medication with no or mild side effects.

In [ ]:
auto_chat = generate_chat("I am looking for something for heart failure, it should work well and I want to avoid harsh side effects.")
auto_text = " ".join(t["text"] for t in auto_chat["conversation"] if t["speaker"].lower() == "customer")
auto_recommendations = recommend_query(auto_text, "embedding", condition="heart failure", top_n=5)
print("Conversation backend:", auto_chat["backend"])
if auto_chat.get("reason"):
    print("Note:", auto_chat["reason"])
display(pd.DataFrame(auto_chat["conversation"]))
display(auto_recommendations)

Conversation backend: gemini-2.5-flash


,speaker,text
0,Assistant,Have you already discussed your heart failure ...
1,Customer,"Yes, I have consulted with my doctor. They've ..."
2,Assistant,Are you currently taking any prescription medi...
3,Customer,I am currently taking a daily multivitamin and...
4,Assistant,Could you describe what specific symptoms you ...
5,Customer,I'm mostly experiencing fatigue and some short...


,item_id,urlDrugName,condition,rating,review_count,effectiveness,sideEffects,final_score
0,792,enalapril,management of congestive heart failure,4.0,1,Highly Effective,Mild Side Effects,0.677542
1,1072,lasix,congestive heart failure and edema,7.0,1,Considerably Effective,Severe Side Effects,0.600285
2,1747,prinivil,? 'heart failure',3.0,1,Ineffective,Moderate Side Effects,0.581860


Although the condition matches in all of the examples, the recommended medication is not as good of a match as in the previous example because this dataset has much less congestive heart failure treatment than for hypertension. This shows a limitation of my project, as the chatbot successfully identifies the user preference, but the quality of the recommendations is limited to the dataset.